# Libraries
## 1. DuckDB

**DuckDB** is an embedded analytical SQL database for Python.
It is optimized for **fast query execution** and **low memory usage**, making it ideal for analytical workloads.
Developers often use it for:

* Data analysis
* Data manipulation
* Reporting tasks

You can find more information on the official website:
🔗 [https://duckdb.org/](https://duckdb.org/)

---

## 2. magic-duckdb

**magic_duckdb** is a Python package that adds Jupyter Notebook *magic commands* for working with DuckDB.
It allows you to run SQL queries directly within a Jupyter Notebook environment, providing a smooth workflow for data exploration and analysis.

Explore its usage and documentation here:

* GitHub repository: [https://github.com/iqmo-org/magic_duckdb](https://github.com/iqmo-org/magic_duckdb)
* PyPI: [https://pypi.org/project/magic-duckdb/](https://pypi.org/project/magic-duckdb/)

---

## 3. Polars

**Polars** is a high-performance DataFrame library built in **Rust** and accessible via **Python**.
It is designed for **big data processing** and provides a user-friendly API similar to Pandas, but with much better scalability and speed.

Key advantages:

* Efficient memory usage
* Parallel computation
* Fast query execution

Learn more at the official website:
🔗 [https://pola.rs/](https://pola.rs/)

---

## 4. Plotly Express

**Plotly Express** is a high-level visualization library built on top of **Plotly**.
It enables the creation of **interactive plots and charts** with minimal code.
You can quickly generate bar charts, scatter plots, line charts, and more without deep configuration.

Official documentation:
🔗 [https://plotly.com/python/plotly-express/](https://plotly.com/python/plotly-express/)

---

## 5. nbformat

**nbformat** is a Python library for working with the **Jupyter Notebook file format**.
It allows developers to:

* Read and write `.ipynb` files
* Modify and extract cell content
* Convert notebooks programmatically

This library is useful for automating Jupyter Notebook operations.

Official documentation and resources:

* Docs: [https://nbformat.readthedocs.io/en/latest/](https://nbformat.readthedocs.io/en/latest/)
* PyPI: [https://pypi.org/project/nbformat/](https://pypi.org/project/nbformat/)

In [25]:
import duckdb
import pandas as pd
import os
import sys
#import plotly_express as px
pd.set_option('display.float_format', '{:.2f}'.format)  # Show 2 decimal places


%load_ext magic_duckdb

The magic_duckdb extension is already loaded. To reload it, use:
  %reload_ext magic_duckdb


#### We already installed and loaded the duckdb magic in our notebook. Let us take advantage of it <br>so that we don't repeate `duckdb.sql` ever time. 

<br>Instead we can use: <br>
- `%dql` for single line queries and:<br>
- `%%dql` for multi-line queries instead


##### Because we are using the magic_duckdb extension, our queries will return a Pandas DataFrame, <br> bringing the entire query result into memory.

* We can avoid this by setting the type of return by using `"-t"` followed by the type, choosing from "df", "arrow", "pl", "describe", "show" and "relation".


#### We can also query files over the internet using duckdb's `httpfs` extension

# 1. Install httpfs
* Install the httpfs extension to enable reading files over HTTP/HTTPS
* This allows DuckDB to query remote files directly from URLs

In [26]:
%%dql -t df
INSTALL httpfs;

Unable to connect, connection may already be closed. Setting connection to None
Traceback (most recent call last):
  File "/Users/minh.pham/personal/project/Learn-DuckDB/.venv/lib/python3.13/site-packages/magic_duckdb/magic.py", line 248, in execute
    o = dbwrapper.execute(
        query_string=query,
    ...<6 lines>...
        export_kwargs=export_kwargs,
    )
  File "/Users/minh.pham/personal/project/Learn-DuckDB/.venv/lib/python3.13/site-packages/magic_duckdb/duckdb_mode.py", line 144, in execute
    r = execute_db(
        query=query_string, con=connection, params=params, execute=execute
    )
  File "/Users/minh.pham/personal/project/Learn-DuckDB/.venv/lib/python3.13/site-packages/magic_duckdb/duckdb_mode.py", line 18, in execute_db
    return con.execute(query, parameters=params)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
_duckdb.ConnectionException: Connection Error: Connection already closed!
While executing query_string='INSTALL httpfs;'


In [27]:
%%dql -t df
LOAD  httpfs;

,Success


# 2. Testing Parquet Performance

In [28]:
%%dql
SELECT format('{:,}', COUNT(*)) as count 
FROM 'https://github.com/cwida/duckdb-data/releases/download/v1.0/taxi_2019_04.parquet'

,count
0,"7,433,139"


In [29]:
%%time
%%dql -t show
SELECT * FROM 'https://github.com/cwida/duckdb-data/releases/download/v1.0/taxi_2019_04.parquet' 
LIMIT 5;

┌───────────┬─────────────────────┬─────────────────────┬─────────────────┬───────────────┬──────────────┬────────────────────┬────────────────────┬─────────────────────┬──────────────┬─────────────┬───────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┐
│ vendor_id │      pickup_at      │     dropoff_at      │ passenger_count │ trip_distance │ rate_code_id │ store_and_fwd_flag │ pickup_location_id │ dropoff_location_id │ payment_type │ fare_amount │ extra │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │
│  varchar  │      timestamp      │      timestamp      │      int8       │     float     │   varchar    │      varchar       │       int32        │        int32        │   varchar    │    float    │ float │  float  │   float    │    float     │         float         │    float     │        float         │
├───────────┼─────────────────────┼─────────────────────┼─────────────────┼─

## 2.1 Performance for query over the internet via httpfs

### Example response

```zsh
CPU times: user 214 ms, sys: 29.6 ms, total: 243 ms
Wall time: 416 ms
```

In [30]:
%%time
%%dql
SELECT COUNT(*) AS RowCount, 
       AVG(passenger_count) AS avg_number_of_passengers,
       AVG(trip_distance) AS avg_trip_distance,
       MAX(trip_distance) AS max_trip_distance,
       AVG(fare_amount) AS avg_fare_amount,
       MAX(fare_amount) AS max_fare_amount,
       AVG(tip_amount) AS avg_tip_amount,
       MAX(tip_amount) AS max_tip_amount 
FROM 'https://github.com/cwida/duckdb-data/releases/download/v1.0/taxi_2019_04.parquet';

CPU times: user 1.77 s, sys: 407 ms, total: 2.18 s
Wall time: 8.16 s


## 2.1 Performance for query local file
### Example response
```zsh
CPU times: user 2.56 ms, sys: 3.98 ms, total: 6.54 ms
Wall time: 6.45 ms
```


In [31]:
%%time
%%dql
SELECT format('{:,}', COUNT(*)) as count FROM '../data/parquet/taxi_2019_04.parquet';

CPU times: user 1.44 ms, sys: 835 μs, total: 2.27 ms
Wall time: 4.85 ms


## 2.3 Example of DuckDB's COLUMNS functions
* Reads from the Parquet file taxi_2019_04.parquet
* Uses DuckDB's COLUMNS() function with a lambda: `c -> c LIKE 'to%'` selects columns whose names start with "to"
* Limits output to 5 rows
* Returns the result as a pandas DataFrame in df

In [32]:
%%dql -t df
SELECT COLUMNS(c -> c LIKE 'to%') FROM '../data/parquet/taxi_2019_04.parquet' limit 5;

,tolls_amount,total_amount
0,0.00,8.80
1,0.00,8.30
2,0.00,47.75
3,0.00,7.30
4,0.00,23.15


In [33]:
%%dql
DESCRIBE FROM '../data/parquet/taxi_2019_04.parquet';

,column_name,column_type,null,key,default,extra
0,vendor_id,VARCHAR,YES,None,None,None
1,pickup_at,TIMESTAMP,YES,None,None,None
2,dropoff_at,TIMESTAMP,YES,None,None,None
3,passenger_count,TINYINT,YES,None,None,None
4,trip_distance,FLOAT,YES,None,None,None
5,rate_code_id,VARCHAR,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,pickup_location_id,INTEGER,YES,None,None,None
8,dropoff_location_id,INTEGER,YES,None,None,None
9,payment_type,VARCHAR,YES,None,None,None


## 3. Parquet file

### 3.1 Let us take a look at the parquet file's metadata

In [34]:
%%dql 
SELECT *
FROM parquet_metadata('../data/parquet/taxi_2019_04.parquet')

,file_name,row_group_id,row_group_num_rows,row_group_num_columns,row_group_bytes,column_id,file_offset,num_values,path_in_schema,type,...,total_compressed_size,total_uncompressed_size,key_value_metadata,bloom_filter_offset,bloom_filter_length,min_is_exact,max_is_exact,row_group_compressed_bytes,geo_bbox,geo_types
0,../data/parquet/taxi_2019_04.parquet,0,122880,18,3398583,0,0,122880,vendor_id,BYTE_ARRAY,...,24900,31341,{},140891876,47,True,True,1,<NA>,<NA>
1,../data/parquet/taxi_2019_04.parquet,0,122880,18,3398583,1,0,122880,pickup_at,INT64,...,584217,983071,{},<NA>,<NA>,True,True,1,<NA>,<NA>
2,../data/parquet/taxi_2019_04.parquet,0,122880,18,3398583,2,0,122880,dropoff_at,INT64,...,611951,983071,{},<NA>,<NA>,True,True,1,<NA>,<NA>
3,../data/parquet/taxi_2019_04.parquet,0,122880,18,3398583,3,0,122880,passenger_count,INT32,...,35942,46317,{},140891923,47,True,True,1,<NA>,<NA>
4,../data/parquet/taxi_2019_04.parquet,0,122880,18,3398583,4,0,122880,trip_distance,FLOAT,...,194815,194798,{},140891970,4112,True,True,1,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1093,../data/parquet/taxi_2019_04.parquet,60,60339,18,1619615,13,0,60339,tip_amount,FLOAT,...,79852,79836,{},141667922,2064,True,True,1,<NA>,<NA>
1094,../data/parquet/taxi_2019_04.parquet,60,60339,18,1619615,14,0,60339,tolls_amount,FLOAT,...,10371,42866,{},141669986,80,True,True,1,<NA>,<NA>
1095,../data/parquet/taxi_2019_04.parquet,60,60339,18,1619615,15,0,60339,improvement_surcharge,FLOAT,...,728,5022,{},141670066,47,True,True,1,<NA>,<NA>
1096,../data/parquet/taxi_2019_04.parquet,60,60339,18,1619615,16,0,60339,total_amount,FLOAT,...,91530,91516,{},141670113,4112,True,True,1,<NA>,<NA>


Let us limit the number of columns to what we are looking for

In [35]:
%%dql 
SELECT file_name, total_compressed_size, total_uncompressed_size 
FROM parquet_metadata('../data/parquet/taxi_2019_04.parquet')

,file_name,total_compressed_size,total_uncompressed_size
0,../data/parquet/taxi_2019_04.parquet,24900,31341
1,../data/parquet/taxi_2019_04.parquet,584217,983071
2,../data/parquet/taxi_2019_04.parquet,611951,983071
3,../data/parquet/taxi_2019_04.parquet,35942,46317
4,../data/parquet/taxi_2019_04.parquet,194815,194798
...,...,...,...
1093,../data/parquet/taxi_2019_04.parquet,79852,79836
1094,../data/parquet/taxi_2019_04.parquet,10371,42866
1095,../data/parquet/taxi_2019_04.parquet,728,5022
1096,../data/parquet/taxi_2019_04.parquet,91530,91516


Let us see how much space the parquet file takes up on disk

In [36]:
%%dql -t df
SELECT 
       format('{:,}', CAST(ROUND((sum(total_compressed_size) / (1024))) AS INT)) as compressed_KB,
       format('{:,}', CAST(ROUND((sum(total_uncompressed_size) / (1024))) AS INT)) as uncompressed_KB,
       format('{:,}', CAST(ROUND((sum(total_compressed_size) / (1024 * 1024))) AS INT)) as compressed_MB,
       format('{:,}', CAST(ROUND((sum(total_uncompressed_size) / (1024 * 1024))) AS INT)) as uncompressed_MB,
       sum(total_compressed_size) / (1024 * 1024 * 1024) as compressed_GB,
       sum(total_uncompressed_size) / (1024 * 1024 * 1024) as uncompressed_GB    
FROM parquet_metadata('../data/parquet/taxi_2019_04.parquet')

,compressed_KB,uncompressed_KB,compressed_MB,uncompressed_MB,compressed_GB,uncompressed_GB
0,"137,590","199,098",134,194,0.13,0.19


Let us look at some aggregates from our data

In [37]:
%%time
%%dql -t show
SELECT format('{:,}', COUNT(*)) AS RowCount, 
       AVG(passenger_count) AS avg_number_of_passengers,
       AVG(trip_distance) AS avg_trip_distance,
       MAX(trip_distance) AS max_trip_distance,
       AVG(fare_amount) AS avg_fare_amount,
       MAX(fare_amount) AS max_fare_amount,
       AVG(tip_amount) AS avg_tip_amount,
       MAX(tip_amount) AS max_tip_amount 
FROM '../data/parquet/taxi_2019_04.parquet';

┌───────────┬──────────────────────────┬────────────────────┬───────────────────┬────────────────────┬─────────────────┬────────────────────┬────────────────┐
│ RowCount  │ avg_number_of_passengers │ avg_trip_distance  │ max_trip_distance │  avg_fare_amount   │ max_fare_amount │   avg_tip_amount   │ max_tip_amount │
│  varchar  │          double          │       double       │       float       │       double       │      float      │       double       │     float      │
├───────────┼──────────────────────────┼────────────────────┼───────────────────┼────────────────────┼─────────────────┼────────────────────┼────────────────┤
│ 7,433,139 │        1.573300727996611 │ 2.9980189376583324 │             830.9 │ 13.194329902362187 │       395839.94 │ 2.2126877947202788 │          440.8 │
└───────────┴──────────────────────────┴────────────────────┴───────────────────┴────────────────────┴─────────────────┴────────────────────┴────────────────┘

CPU times: user 136 ms, sys: 17.8 ms, total: 

 # 4. Variables

Notice the extreme fare_amount value ($395,839.94)
We need to inspect that record. It could be a corrupt record.

In [38]:
%%dql 
SELECT * FROM '../data/parquet/taxi_2019_04.parquet' WHERE fare_amount = 
(SELECT MAX(fare_amount) FROM '../data/parquet/taxi_2019_04.parquet');

,vendor_id,pickup_at,dropoff_at,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
0,1,2019-04-29 03:45:58,2019-04-29 03:46:33,1,0.00,1,N,145,145,2,395839.94,0.50,0.50,0.00,0.00,0.30,395841.25,0.00


<i>As we suspected there is a problem with paying that fare_amount for a trip_distance of 0.00 miles. <br>Also notice the pickup_at, dropoff_at timestamps as well as pickup_location_id and dropoff_location_id.</i>

If you wish to merge duckdb sql queries and python code, you should use the duckdb.sql('...') query method, 

In [39]:
var1 = duckdb.sql(" SELECT COUNT(*) FROM '../data/parquet/taxi_2019_04.parquet' WHERE pickup_at BETWEEN '2019-04-10' AND '2019-04-12'");
print(var1)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       535737 │
└──────────────┘



or use the `%dql -o <variablename>` method.

In [40]:
%%dql -o var2 
SELECT COUNT(*) FROM '../data/parquet/taxi_2019_04.parquet' WHERE pickup_at BETWEEN '2019-04-10' AND '2019-04-12'

,count_star()
0,535737


In [41]:
print(var2)

   count_star()
0        535737


# 5. More magic_duckdb

List available DQL output types

The `%dql --listtypes` command shows all available output formats for DQL query results:
- `df`: Returns a pandas DataFrame (default)
- `df_markdown`: Returns a pandas DataFrame formatted as markdown
- `markdown`: Returns the result as markdown text
- `arrow`: Returns an Apache Arrow table
- `fetch_arrow_table`: Fetches the result as an Arrow table
- `pl`: Returns a Polars DataFrame
- `describe`: Returns schema information about the result
- `show`: Displays the result (similar to SHOW in SQL)
- `relation`: Returns a DuckDB relation object for further manipulation

In [42]:
%dql --listtypes

['df',
 'df_markdown',
 'markdown',
 'arrow',
 'fetch_arrow_table',
 'pl',
 'describe',
 'show',
 'relation']

List all tables used in the query

In [43]:
%%dql --tables  
SELECT COUNT(*) As qCount FROM '../data/parquet/taxi_2019_04.parquet'

{'../data/parquet/taxi_2019_04.parquet'}

Get the connection created within DQL and use it directly

In [44]:
con = %dql --getcon
display(con.sql("pragma version").df())

,library_version,source_id,codename
0,v1.4.1,b390a7c376,Andium


In [45]:
con = duckdb.connect("../database/taxi_2019_04.db") 
con.close()

Create a connection explicitly, and pass it explicitly to the dql connection. <br>dql by default will use the duckdb default connection

Create a table from a our taxi_2019_04 Parquet file

In [46]:
con = duckdb.connect("../database/taxi_2019_04.db")
con.execute("CREATE OR REPLACE TABLE taxi_trips AS (SELECT * FROM '../data/parquet/taxi_2019_*.parquet');")

The `-co` sets the connection to an existing database object

In [47]:
%dql -co con
%dql SELECT format('{:,}', COUNT(*)) AS count FROM taxi_trips;

,count
0,"21,939,424"


We can also convert to pandas dataFrame directly from your duckdb query<br> by adding the ".df()" function

Query the first 10 rows of the newly created taxi_trips data. <br>

<i>Notice that I use duckdb's to pandas dataframe `df()` at the end of the query.</i>


In [48]:
con.sql("SELECT * FROM taxi_trips LIMIT 3").df()

,vendor_id,pickup_at,dropoff_at,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
0,1,2019-04-01 00:04:09,2019-04-01 00:06:35,1,0.50,1,N,239,239,1,4.00,3.00,0.50,1.00,0.00,0.30,8.80,2.50
1,1,2019-04-01 00:22:45,2019-04-01 00:25:43,1,0.70,1,N,230,100,2,4.50,3.00,0.50,0.00,0.00,0.30,8.30,2.50
2,1,2019-04-01 00:39:48,2019-04-01 01:19:39,1,10.90,1,N,68,127,1,36.00,3.00,0.50,7.95,0.00,0.30,47.75,2.50


# 6. Analyzing Data with DuckDB

## 6.1 SQL queries using DuckDB:

To run SQL queries in DuckDB we can directly use ".sql", no need to create a connection to ":memory:"

Every DataFrame inside this notebook will be instantly available for DuckDB to make SQL queries against.


In [53]:
%dql SUMMARIZE SELECT * FROM taxi_trips WHERE vendor_id = 1;

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,vendor_id,VARCHAR,1,1,1,None,None,None,None,None,8078672,0.00
1,pickup_at,TIMESTAMP,2019-04-01 00:00:02,2019-06-30 23:59:56,6139537,2019-05-15 08:18:39.560889,None,2019-04-22 16:34:27.359958,2019-05-14 14:02:03.527029,2019-06-06 18:38:07.325822,8078672,0.00
2,dropoff_at,TIMESTAMP,2018-09-13 10:38:20,2019-07-01 01:01:07,6232947,2019-05-15 08:33:15.556128,None,2019-04-22 16:03:50.773555,2019-05-14 17:00:30.827078,2019-06-06 16:37:42.211914,8078672,0.00
3,passenger_count,TINYINT,0,9,11,1.2334254936950033,0.6986199549033979,1,1,1,8078672,0.00
4,trip_distance,FLOAT,0.0,830.9,566,2.90599940178197,3.8033649619023255,0.9544353491269193,1.5976895302214933,2.9771739279852816,8078672,0.00
5,rate_code_id,VARCHAR,1,99,7,None,None,None,None,None,8078672,0.00
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,8078672,0.00
7,pickup_location_id,INTEGER,1,265,290,164.02025704224656,66.16504804891491,120,162,234,8078672,0.00
8,dropoff_location_id,INTEGER,1,265,290,161.72963910405076,69.99313217245532,111,162,234,8078672,0.00
9,payment_type,VARCHAR,1,4,4,None,None,None,None,None,8078672,0.00


#### Let us look at the Datatypes in the table

In [54]:
%dql -co con
%dql DESCRIBE taxi_trips;

,column_name,column_type,null,key,default,extra
0,vendor_id,VARCHAR,YES,None,None,None
1,pickup_at,TIMESTAMP,YES,None,None,None
2,dropoff_at,TIMESTAMP,YES,None,None,None
3,passenger_count,TINYINT,YES,None,None,None
4,trip_distance,FLOAT,YES,None,None,None
5,rate_code_id,VARCHAR,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,pickup_location_id,INTEGER,YES,None,None,None
8,dropoff_location_id,INTEGER,YES,None,None,None
9,payment_type,VARCHAR,YES,None,None,None


#### Let us test out query speed in the new duckdb table

In [55]:
%%time
%%dql
SELECT format('{:,}', COUNT(*)) AS RowCount, 
       AVG(passenger_count) AS avg_number_of_passengers,
       AVG(trip_distance) AS avg_trip_distance,
       MAX(trip_distance) AS max_trip_distance,
       AVG(fare_amount) AS avg_fare_amount,
       MAX(fare_amount) AS max_fare_amount,
       AVG(tip_amount) AS avg_tip_amount,
       MAX(tip_amount) AS max_tip_amount FROM taxi_trips;

CPU times: user 388 ms, sys: 4.73 ms, total: 393 ms
Wall time: 40.8 ms


,RowCount,avg_number_of_passengers,avg_trip_distance,max_trip_distance,avg_fare_amount,max_fare_amount,avg_tip_amount,max_tip_amount
0,"21,939,424",1.57,3.04,45977.22,13.43,395839.94,2.25,1624.64
